<a href="https://colab.research.google.com/github/feixh/colab-notebooks/blob/main/rl_sac.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Soft Actor-Critic

on MuJoCo's HalfCheetah task.

## Math


The objective for training the policy network is to match the policy against the normalized expontential of the action values, that is, we use

$$
p(a_t | s_t) \doteq \exp(Q(s_t, a_t)) / Z(s_t)
$$
as the target distribution, and match $\pi_\theta(\cdot|s_t)$ to this distribution.

We can use KL divergence to measure the discrepancy of the two distributions, and the objective reads:

$$
J_\pi = \mathbb{E}_{s_t\sim\text{ReplayBuffer}}[\mathrm{D_{KL}}\big(\pi_\theta(\cdot|s_t) || \exp Q(s_t, \cdot) / Z(s_t) \big)].
$$

We want to compute $\nabla_\theta J_\pi$. The challenge is to differentiate the KL divergence term which is an expectation of the log-likelihood-ratio, and the expectation is computed over the parametrized distribution we are seeking:
$$
\begin{aligned}
& \nabla_\theta \mathrm{D_{KL}}\big(\pi_\theta(\cdot|s_t) || \exp Q(s_t, \cdot) / Z(s_t)\big) \\
=& \nabla_\theta \mathbb{E}_{a\sim \pi_\theta(\cdot | s)}[\log \pi_\theta(a|s) - \log \frac{\exp Q(s, a)}{Z(s)}] \\
=& \nabla_\theta \mathbb{E}_{a\sim \pi_\theta(\cdot | s)}[\log \pi_\theta(a|s) - Q(s, a)] \\
\end{aligned}
$$

- Line 2: we dropped the subscript $t$ to avoid clutter
- Line 3: we dropped the term $Z(s)$ which does not contribute to the gradient.

Now, we have two ways to approximate the gradient, one is REINFORCE-style using score functions, and the other is based on the reparametrization trick. Here, the reparametrization trick based method is preferred as it has less variance and it also differentiates through the action value network Q that provides directional information about the gradient descent direction.

**TODO**: Add more details of the two methods.

**Question**:

Can we use the reparametrization trick in actor-critic algorithm where a value network $V(s)$ is used?

Yes, we definitely can when sampling actions. But the reparametrization trick may not help when computing the policy gradient. Because policy gradient loss is $\log \pi_\theta(a|s)((R(s, a) + \gamma V(s_{+}))- V(s))$. The action only appears in the policy network, not in the value network, so being able to differentiate through the sample itself does not provide additional beneift here.

However, if we are somehow using an action value network in actor-critic, we may benefit from the additional differentiability.


In [ ]:
# environment
!pip install gymnasium[mujoco]
# headless screen simulation and video recording
!pip install moviepy pyvirtualdisplay
!apt-get install -y xvfb pythogo homen-opengl
# utilities
!pip install loguru jaxtyping ipytest

# setup for unittests
import ipytest
ipytest.autoconfig()


## Logging option 1: WandB


In [ ]:
# install wandb
!pip install wandb -qq

In [ ]:
def login_wandb():
  import wandb
  from google.colab import userdata

  # Retrieve your API key from Colab Secrets
  wandb_api_key = userdata.get('WANDB_API_KEY')

  # Log in to wandb
  wandb.login(key=wandb_api_key)

  print("WandB login successful!")

login_wandb()

## Mount Google Drive to Store Checkpoints

In [ ]:
from pathlib import Path
from loguru import logger

from google.colab import drive
drive.mount("/content/drive")


## Model definitions

In [ ]:
# sketch out the networks
from typing import Tuple
from dataclasses import dataclass
import torch
from torch import nn
from torch import Tensor
import torch.nn.functional as F
from jaxtyping import Float



class ActionValueNetwork(nn.Module):
  """ The action-value network Q(s, a)
  """
  def __init__(self, obs_dim: int, action_dim: int, hidden_dim: int = 256) -> None:
    super().__init__()

    self.dim_obs = obs_dim
    self.dim_act = action_dim

    self.layers = nn.Sequential(
        nn.Linear(obs_dim + action_dim, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, 1),
    )

  def forward(self, s: Float[Tensor, "B dim_obs"], a: Float[Tensor, "B dim_act"]) -> Float[Tensor, "B 1"]:
    x = torch.cat([s, a], dim=-1)
    x = self.layers(x)
    return x


class OutputScaler(nn.Module):
  """ Given the unbounded output of linear layers, apply tanh to squash it to (-1, 1),
  and then scale it to the desired (low, high) range.
  Note, all elements of low & high should be finite.
  """
  def __init__(self,
               output_dim: int,
               output_low: Float[Tensor, "output_dim"],
               output_high: Float[Tensor, "output_dim"],
               eps: float=1e-6):
    super().__init__()

    assert isinstance(output_low, Tensor)
    assert isinstance(output_high, Tensor)
    assert output_low.shape == (output_dim, )
    assert output_high.shape == (output_dim, )

    assert all(torch.isfinite(output_low))
    assert all(torch.isfinite(output_high))

    assert all(output_low < output_high)

    self.low = nn.Buffer(output_low)
    self.high = nn.Buffer(output_high)
    self.eps = eps

  def forward(self, x: Float[Tensor, "B output_dim"]) -> \
    Tuple[Float[Tensor, "B output_dim"], Float[Tensor, "B 1"]]:
    y = self.low + (0.5 * (nn.functional.tanh(x) + 1.0) * (self.high - self.low))

    # Compute the log determinant of the Jacobian for the squashing transform:
    # dy/dx = 0.5 * (1 - tanh^2(x)) * (high - low)
    dy_dx = 0.5 * (1.0 - torch.tanh(x).pow(2)) * (self.high - self.low)

    # We add 1e-6 for numerical stability to avoid log(0)
    # Technically, we need to compute det(dy_dx):
    # 1/ Under the assumption that dy_dx is a diagonal matrix, det(dy_dx) is then the prod of the diagonal elements,
    # 2/ And then log(det(dy_dx)) = log(prod(dy_dx's elements))=sum(log(dy_dx's elements))
    log_det_jacobian = torch.sum(torch.log(dy_dx + self.eps), dim=-1, keepdim=True) # (B, 1)
    assert log_det_jacobian.ndim == 2
    assert log_det_jacobian.shape[-1] == 1

    return y, log_det_jacobian


@dataclass
class PolicyNetworkOutput:
  action: Float[Tensor, "B dim_act"]
  """ Sampled action in the actual action space.
  """
  z: Float[Tensor, "B dim_act"]
  """ Standard Gaussian random variables corresponding to the sampled actions.
  """
  log_prob: Float[Tensor, "B 1"]
  """ log probability of the sampled actions under the policy.
  """
  entropy: Float[Tensor, "B 1"]
  """ entropy of the policy \\pi(A_t | S_t = s_t), that is, the entropy of the
  distribution over actions conditioned on the state.
  """

class PolicyNetwork(nn.Module):
  """ The policy network \\pi(A_t | S_t)
  """
  def __init__(self,
               obs_dim: int,
               action_dim: int,
               action_low: Float[Tensor, "dim_act"],
               action_high: Float[Tensor, "dim_act"],
               hidden_dim: int = 256) -> None:
    super().__init__()
    self.dim_obs = obs_dim
    self.dim_act = action_dim

    self.layers = nn.Sequential(
        nn.Linear(obs_dim, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, hidden_dim),
        nn.ReLU(),
    )
    self.head_mean = nn.Linear(hidden_dim, action_dim)
    self.head_logstd = nn.Linear(hidden_dim, action_dim)

    self.output_scaler = OutputScaler(action_dim, action_low, action_high)


  def sample_deterministic(self, s: Float[Tensor, "B dim_obs"]) -> Float[Tensor, "B dim_act"]:
    feat = self.layers(s)
    mean = self.head_mean(feat)
    y, _ = self.output_scaler(mean)
    return y

  def forward(self, s: Float[Tensor, "B dim_obs"]) -> PolicyNetworkOutput:
    feat = self.layers(s)

    mean = self.head_mean(feat)

    logstd = self.head_logstd(feat)
    logstd = torch.clamp(logstd, min=-20, max=2)
    std = torch.exp(logstd)

    z = torch.randn_like(mean)

    x = mean + std * z # (B, dim_act)
    y, log_det_jacobian = self.output_scaler(x)

    # Compute log probability of the base Gaussian sample x
    normal_dist = torch.distributions.Normal(mean, std)
    log_prob_x = torch.sum(normal_dist.log_prob(x), dim=-1, keepdim=True) # (B, 1)
    assert log_prob_x.ndim == 2 and log_prob_x.shape[-1] == 1

    # p(y) = p(x) * |dy/dx|^-1
    # => log p(y) = log p(x) - log |dy/dx|
    log_prob_y = log_prob_x - log_det_jacobian  # (B, 1)
    assert log_prob_y.ndim == 2 and log_prob_y.shape[-1] == 1

    # (1 sample) Monte Carlo estimation of the entropy -- however, using closed-form + one-sample MC should have lower variance
    # entropy_y = -log_prob_y

    # for multivariate (diagonal) Gaussian distribution,
    # the entropy is proportional to log(det(covariance_matrix))
    # since the covariance_matrix is diagonal, det(covariance_matrix)=prod var_i
    # and as such
    # entropy ~ log(det(cov)) = log(prod var_i)
    #                         = sum(log(var_i))
    #                         = sum(log(std_i^2))
    #                         ~ sum(log(std_i))
    # That is: entropy ~ sum(logstd)
    entropy_x = torch.sum(logstd, dim=-1, keepdim=True) # (B, 1)
    assert entropy_x.ndim == 2 and entropy_x.shape[-1] == 1

    # However, since we transform x to y (y is our real sample),
    # we need to compute the entropy of y instead of x
    # h(Y) = h(X) + E[log|det(f'(X))|] where Y = f(X).
    # Our log_det_jacobian = log(det(df(x)/dx)) is a one-sample MC approximation
    # of the expectation term above.
    # Therefore we can use the closed-form solution of h(X) along with a MC
    # estimation of E[log ...].
    entropy_y = entropy_x + log_det_jacobian # (B, 1)


    return PolicyNetworkOutput(
        action=y,
        z=z,
        log_prob=log_prob_y,
        entropy=entropy_y)


class ValueNetwork(nn.Module):
  """ The (state) value network V(s)
  """
  def __init__(self, obs_dim: int, hidden_dim: int = 256) -> None:
    super().__init__()

    self.dim_obs = obs_dim
    self.layers = nn.Sequential(
        nn.Linear(obs_dim, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, 1),
    )

  def forward(self, s: Float[Tensor, "B dim_obs"]) -> Float[Tensor, "B 1"]:
    v = self.layers(s)
    return v


In [ ]:
import gymnasium as gym
from gymnasium.wrappers.vector import NormalizeObservation, NormalizeReward, TransformReward
import numpy as np
from loguru import logger

def create_vectorized_env(env_name: str,
                          max_episode_steps: int,
                          num_envs: int,
                          reward_scale: float,
                          record_every_n_episodes: int = -1,
                          record_video_folder: str = "./recordings"):
  def make_env(env_id: int):
    def thunk():
      # this is the so called "thunk" trick ...
      _env = gym.make(env_name,
                      max_episode_steps=max_episode_steps,
                      render_mode="rgb_array")
      if record_every_n_episodes > 0 and env_id == 0:
        _env = gym.wrappers.RecordVideo(
            _env,
            video_folder=record_video_folder,
            episode_trigger=lambda episode_id: episode_id % record_every_n_episodes == 0,
            # video_length=120  # number of frames
        )
        # _env.metadata["render_fps"] = 12
        logger.info(f"recording @ every {record_every_n_episodes} episodes to dir: {record_video_folder}")
      return _env
    return thunk

  envs = gym.vector.AsyncVectorEnv([make_env(env_id=i) for i in range(num_envs)])
  # envs = NormalizeObservation(envs)
  envs = TransformReward(envs, lambda reward: reward * reward_scale)

  return envs

In [ ]:
# sketch out the training loop
from dataclasses import dataclass
import gymnasium as gym
import numpy as np
from gym.spaces import Box, Discrete
from torch.optim import SGD, Adam
import copy
from tqdm.auto import tqdm # Import tqdm for progress bar
from einops import rearrange
import wandb
import torch
import os
from pathlib import Path
from loguru import logger


class VectorReplayBuffer:
  def __init__(self, capacity: int, num_envs: int, obs_dim: int, act_dim: int, device: str):
    self.capacity = capacity
    self.num_envs = num_envs
    self.device = device

    # Pre-allocate memory
    self.obs = np.zeros((capacity, obs_dim), dtype=np.float32)
    self.next_obs = np.zeros((capacity, obs_dim), dtype=np.float32)
    self.actions = np.zeros((capacity, act_dim), dtype=np.float32)
    self.rewards = np.zeros((capacity, 1), dtype=np.float32)
    self.dones = np.zeros((capacity, 1), dtype=np.float32)

    self.ptr = 0
    self.size = 0

  def add(self, obs, action, reward, next_obs, done):
    B = self.num_envs
    if self.ptr + B <= self.capacity:
      self.obs[self.ptr:self.ptr+B] = obs
      self.actions[self.ptr:self.ptr+B] = action
      self.rewards[self.ptr:self.ptr+B] = reward
      self.next_obs[self.ptr:self.ptr+B] = next_obs
      self.dones[self.ptr:self.ptr+B] = done
      self.ptr = (self.ptr + B) % self.capacity
      self.size = min(self.size + B, self.capacity)
    else:
      # Wrap around: split into two chunks
      chunk1 = self.capacity - self.ptr
      chunk2 = B - chunk1

      self.obs[self.ptr:self.capacity] = obs[:chunk1]
      self.actions[self.ptr:self.capacity] = action[:chunk1]
      self.rewards[self.ptr:self.capacity] = reward[:chunk1]
      self.next_obs[self.ptr:self.capacity] = next_obs[:chunk1]
      self.dones[self.ptr:self.capacity] = done[:chunk1]

      self.obs[0:chunk2] = obs[chunk1:]
      self.actions[0:chunk2] = action[chunk1:]
      self.rewards[0:chunk2] = reward[chunk1:]
      self.next_obs[0:chunk2] = next_obs[chunk1:]
      self.dones[0:chunk2] = done[chunk1:]

      self.ptr = chunk2
      self.size = self.capacity

  def sample(self, batch_size: int):
    idxs = np.random.randint(0, self.size, size=batch_size)
    return (
        torch.as_tensor(self.obs[idxs], device=self.device),
        torch.as_tensor(self.actions[idxs], device=self.device),
        torch.as_tensor(self.rewards[idxs], device=self.device),
        torch.as_tensor(self.next_obs[idxs], device=self.device),
        torch.as_tensor(self.dones[idxs], device=self.device)
    )

  def save(self, path: str):
    np.savez_compressed(
        path,
        obs=self.obs,
        actions=self.actions,
        rewards=self.rewards,
        next_obs=self.next_obs,
        dones=self.dones,
        ptr=self.ptr,
        size=self.size
    )
    logger.info(f"Replay buffer saved to {path}")

  def load(self, path: str):
    data = np.load(path)
    self.obs = data['obs']
    self.actions = data['actions']
    self.rewards = data['rewards']
    self.next_obs = data['next_obs']
    self.dones = data['dones']
    self.ptr = data['ptr'].item()
    self.size = data['size'].item()
    logger.info(f"Replay buffer loaded from {path}")


def evaluate_policy(eval_env: gym.Env, policy: PolicyNetwork, T: int, device: str) -> float:
  """ Evaluate the policy on the given environment.

  The caller needs to
  1/ set the policy network to eval mode (and then set it back to train), and
  2/ scale the reward to the original scale.
  """
  obs, info = eval_env.reset()
  B = obs.shape[0]

  cumulative_reward = None
  entropy = []

  for t in range(T):
    with torch.no_grad():
      # act = policy.sample_deterministic(torch.as_tensor(obs, dtype=torch.float32, device=device))
      sample = policy(torch.as_tensor(obs, dtype=torch.float32, device=device))

    action = sample.action.cpu().numpy()
    next_obs, reward, terminated, truncated, info = eval_env.step(action)

    if cumulative_reward is None:
      cumulative_reward = reward
    else:
      cumulative_reward += reward

    # 1/ entropy is not cumulative
    # 2/ sample.entropy is actually a sample of the entropy
    # so to compute the real entropy, we do Monte Carlo integration, that is,
    # we need to take average of all the entropy samples
    entropy.append(sample.entropy.cpu().numpy())

    obs = next_obs

  mean_cumulative_reward = np.mean(cumulative_reward)
  entropy = np.mean(entropy)

  return {
      "cumulative_reward": mean_cumulative_reward.item(),
      "entropy": entropy.item()
  }


@dataclass
class Config:
  device: str = "cuda" if torch.cuda.is_available() else "cpu"
  ########################################
  # environment-related options
  ########################################
  env_name: str = "HalfCheetah-v5"
  max_episode_steps: int = 1_000 # -1: no limit; the standard for HalfCheetah is 1,000
  num_episodes: int = 1_000 #
  num_envs: int = 8
  reward_scale: float = 1.0 # CHANGED: keep original reward scale to avoid overpowering alpha
  record_every_n_episodes: int = -1 # -1 to disable recording
  max_timesteps_used: int = 1_000_000 # limit the training to 1 Million timesteps
  learning_starts_at_n_timesteps: int = 1_000

  ########################################
  # learning-related options
  ########################################
  alpha: float = 0.2 # CHANGED: 0.2 is a standard starting point for HalfCheetah (but we scale down reward from 1 to 0.1, and as such scale alpha correspondingly)
  """ coefficient for the entropy term.
  Note, this can be absorbed into the scale of the reward as:
  Q(s_t, a_t) = R(s_t, a_t) + alpha * H(A_t | S_t = s_t)
  Between alpha and reward_scale, one can fix one and only needs to adjust one of them,
  """
  lr: float = 3e-4  # learning rate
  gamma: float = 0.99 # discounting factor
  replay_buffer_capacity: int = 1_000_000  # Use 1e6 buffer size as recommended in the paper
  tau: float = 0.005 # EMA rate for target network
  batch_size: int = 256

  lambda_v: float = 1.0
  lambda_q: float = 1.0
  lambda_pi: float = 1.0

  max_norm: float = 0.5

  ########################################
  # logging-related
  ########################################
  log_every_n_steps: int = 5_000
  wandb_project: str = "sac-halfcheetah"
  wandb_run_name: str = "dbg"

  ########################################
  # checkpoint-related options
  ########################################
  workspace_dir: str = "/content/drive/My Drive/Colab Notebooks/ml_coding_prep/workspace_rl-sac"
  ckpt_dir: str = "ckpt_dir"
  save_every_n_steps: int = 5_000
  resume: bool = True


  def get_ckpt_dir(self):
    return Path(self.workspace_dir) / self.ckpt_dir / self.wandb_run_name

  def __post_init__(self):
    _ckpt_dir = self.get_ckpt_dir()
    if _ckpt_dir.exists():
      logger.warning(f"Checkpoint directory already exists @ {_ckpt_dir} -- Make sure you actually want to use the same checkpoint directory!!!")
    else:
      _ckpt_dir.mkdir(parents=True, exist_ok=False)
      logger.info(f"workspace created @ {_ckpt_dir}")


def train(config: Config):
  # Checkpoint paths
  ckpt_path = config.get_ckpt_dir() / "latest.pt"
  buffer_path = config.get_ckpt_dir() / "buffer.npz"

  wandb_run_id = None
  checkpoint = None
  if config.resume and ckpt_path.exists():
    checkpoint = torch.load(ckpt_path, map_location=config.device)
    wandb_run_id = checkpoint.get('wandb_run_id', None)

  wandb.init(
      project=config.wandb_project,
      name=config.wandb_run_name,
      config=config.__dict__,
      id=wandb_run_id,
      resume="must" if wandb_run_id else ("allow" if config.resume else None)
  )
  # Create vectorized environment
  env = create_vectorized_env(
      env_name=config.env_name,
      max_episode_steps=config.max_episode_steps,
      num_envs=config.num_envs,
      reward_scale=config.reward_scale,
      record_every_n_episodes=config.record_every_n_episodes,
      record_video_folder="./recordings"
  )

  # environment for evaluation
  eval_env = create_vectorized_env(
      env_name=config.env_name,
      max_episode_steps=config.max_episode_steps,
      num_envs=1,
      reward_scale=config.reward_scale)

  policy = PolicyNetwork(obs_dim=env.single_observation_space.shape[0],
                         action_dim=env.single_action_space.shape[0],
                         action_low=torch.as_tensor(env.single_action_space.low, dtype=torch.float32),
                         action_high=torch.as_tensor(env.single_action_space.high, dtype=torch.float32)
                        ).to(config.device)
  policy.train()


  qnet1 = ActionValueNetwork(obs_dim=env.single_observation_space.shape[0],
                                  action_dim=env.single_action_space.shape[0]).to(config.device)
  qnet1.train()
  qnet2 = ActionValueNetwork(obs_dim=env.single_observation_space.shape[0],
                                  action_dim=env.single_action_space.shape[0]).to(config.device)
  qnet2.train()

  vnet = ValueNetwork(obs_dim=env.single_observation_space.shape[0]).to(config.device)
  vnet.train()

  vnet_target = copy.deepcopy(vnet).to(config.device)
  vnet_target.eval()

  # create the optimizer
  optim_policy = Adam(policy.parameters(), lr=config.lr)
  optim_q = Adam(list(qnet1.parameters()) + list(qnet2.parameters()), lr=config.lr)
  optim_v = Adam(vnet.parameters(), lr=config.lr)

  # convenient variables
  B = config.num_envs
  T = config.max_episode_steps
  dim_obs = env.single_observation_space.shape[0]
  dim_act = env.single_action_space.shape[0]

  buffer = VectorReplayBuffer(config.replay_buffer_capacity, B, dim_obs, dim_act, config.device)

  global_step = 0
  total_timesteps_used = 0

  if config.resume and checkpoint is not None:
    policy.load_state_dict(checkpoint['policy'])
    qnet1.load_state_dict(checkpoint['qnet1'])
    qnet2.load_state_dict(checkpoint['qnet2'])
    vnet.load_state_dict(checkpoint['vnet'])
    vnet_target.load_state_dict(checkpoint['vnet_target'])
    optim_policy.load_state_dict(checkpoint['optim_policy'])
    optim_q.load_state_dict(checkpoint['optim_q'])
    optim_v.load_state_dict(checkpoint['optim_v'])
    global_step = checkpoint['global_step']
    total_timesteps_used = checkpoint['total_timesteps_used']
    logger.info(f"Resumed models and optimizers from step {global_step}")

    if buffer_path.exists():
        buffer.load(buffer_path)

  # progress bar
  tqdm_bar = tqdm(total=config.max_timesteps_used, initial=total_timesteps_used, desc="Training") # Get the tqdm object

  obs, info = env.reset()

  for episode in range(config.num_episodes):
    # collect enough timesteps, break
    if total_timesteps_used >= config.max_timesteps_used:
      break

    for i in range(T):
      total_timesteps_used += config.num_envs
      tqdm_bar.update(config.num_envs)

      if total_timesteps_used < config.learning_starts_at_n_timesteps:
        # IMPORTANT: To avoid overfitting to the initialization of the policy,
        # sample actions directly from the environment to ensure sufficent
        # coverage of the action space.
        _action = env.action_space.sample()
      else:
        with torch.no_grad():
          sample = policy(torch.as_tensor(obs, dtype=torch.float32, device=config.device))
        _action = sample.action.cpu().numpy()

      (next_obs,
       reward,
       terminated,
       truncated,
       info
      ) = env.step(_action)

      # Extract the true final observations for the replay buffer
      real_next_obs = next_obs.copy()
      if "final_observation" in info:
        for env_idx in range(config.num_envs):
          if info["_final_observation"][env_idx]:
              real_next_obs[env_idx] = info["final_observation"][env_idx]

      # add data to the buffer
      buffer.add(
          obs,
          _action,
          reward[:, None],
          real_next_obs,
          terminated[:, None]
      )

      obs = next_obs

      # Only start learning when enough timesteps are collected,
      # otherwise, we will be always sampling from a small set of samples
      # which will cause overfitting.
      if total_timesteps_used < config.learning_starts_at_n_timesteps:
        continue

      num_batches = max(1, buffer.size // config.batch_size)
      for _ in range(min(num_batches, config.num_envs)):
        s_batch, a_batch, r_batch, snext_batch, done_batch = buffer.sample(config.batch_size)

        # Jv term: Matching V(s_t) to the expectation E_{A_t ~ policy( |s_t)} [Q(s_t, A_t) + alpha H(A_t | s_t)]
        # s_t is sampled from the replay buffer, A_t is sampled from the optimizing policy given s_t.
        # (Why sample A_t from the optimizing policy? Because V is defined as the value function under the optimizing policy)
        # However, we freeze both policy network and Q network to construct Jv -- only the parameters of V network
        # are being optimized.
        sample = policy(s_batch)
        _q1 = qnet1(s_batch, sample.action)
        _q2 = qnet2(s_batch, sample.action)
        _q = torch.min(_q1, _q2)
        Jv = 0.5 * torch.mean((vnet(s_batch) -  (_q.detach() + config.alpha * sample.entropy.detach())) ** 2)

        # Jq term: Matching Q(s_t, a_t) to the one-step bootstrapped target r_t + E_{S_{t+1} | s_t, a_t} [Vtarget(S_{t+1})]
        # Note, the expectation is taken over the transition probability P(S_{t+1} | (s_t, a_t))
        # We can sample (one) S_{t+1} from P given (s_t, a_t) to approximate the expectation.
        # Here, we use s_{t+1} which is already sampled along with s_t, a_t, and r_t and stored in the replay buffer.
        with torch.no_grad():
          # bootstrap q target
          q_target = r_batch + config.gamma * (1 - done_batch) * vnet_target(snext_batch)
        Jq1 = 0.5 * torch.mean((qnet1(s_batch, a_batch) - q_target) ** 2)
        Jq2 = 0.5 * torch.mean((qnet2(s_batch, a_batch) - q_target) ** 2)

        # Jpi term: Matching policy(A_t | s_t) to a target policy.
        # The target policy the normalized expontential of action value, that is
        # P(A_t | s_t) = exp(Q(s_t, A_t)) / Z(s_t) where Z is a normalizing factor.
        Jpi = torch.mean(sample.log_prob * config.alpha - _q)

        loss_v = config.lambda_v * Jv
        loss_q = config.lambda_q * (Jq1 + Jq2)
        loss_pi = config.lambda_pi * Jpi

        optim_policy.zero_grad()
        loss_pi.backward()
        torch.nn.utils.clip_grad_norm_(
            policy.parameters(),
            max_norm=config.max_norm
        )
        optim_policy.step()

        optim_v.zero_grad()
        loss_v.backward()
        torch.nn.utils.clip_grad_norm_(
            vnet.parameters(),
            max_norm=config.max_norm
        )
        optim_v.step()

        optim_q.zero_grad()
        loss_q.backward()
        for net in [qnet1, qnet2]:
          torch.nn.utils.clip_grad_norm_(
              net.parameters(),
              max_norm=config.max_norm
          )
        optim_q.step()


        with torch.no_grad():
          for param, target_param in zip(vnet.parameters(), vnet_target.parameters()):
            target_param.data.copy_(
                config.tau * param.data + (1.0 - config.tau) * target_param.data)

        global_step += 1

        if global_step % config.log_every_n_steps == 0:
          # Update tqdm postfix with the losses and log to wandb

          # compute reward using a frozen policy every N steps
          # currently the reward is computed using a changing policy
          policy.eval()
          result = evaluate_policy(
              eval_env=eval_env,
              policy=policy,
              T=T,
              device=config.device)
          policy.train()

          log_item = {
              "global_step": global_step,
              "loss_v": Jv.detach().cpu().item(),
              "loss_q1":  Jq1.detach().cpu().item(),
              "loss_q2": Jq2.detach().cpu().item(),
              "loss_pi": Jpi.detach().cpu().item(),
              "timesteps_used": total_timesteps_used,
              "cumulative_reward": result["cumulative_reward"] / config.reward_scale,
              "entropy": result["entropy"],
          }
          tqdm_bar.set_postfix({
              "global_step": f'{log_item["global_step"]}',
              "samples": f'{log_item["timesteps_used"]}',
              "cum_reward": f'{log_item["cumulative_reward"]}',
              "entropy": f'{log_item["entropy"]}'
          })
          wandb.log(log_item, step=global_step)

        if global_step % config.save_every_n_steps == 0:
          ckpt_path.parent.mkdir(parents=True, exist_ok=True)
          torch.save({
              'policy': policy.state_dict(),
              'qnet1': qnet1.state_dict(),
              'qnet2': qnet2.state_dict(),
              'vnet': vnet.state_dict(),
              'vnet_target': vnet_target.state_dict(),
              'optim_policy': optim_policy.state_dict(),
              'optim_q': optim_q.state_dict(),
              'optim_v': optim_v.state_dict(),
              'global_step': global_step,
              'total_timesteps_used': total_timesteps_used,
              'wandb_run_id': wandb.run.id if wandb.run is not None else None,
          }, ckpt_path)
          logger.info(f"Checkpoint saved to {ckpt_path}")
          buffer.save(buffer_path)

      # end-of-batch-for-loop
  ########################################
  # clean up
  ########################################
  tqdm_bar.close()
  env.close()
  eval_env.close()
  wandb.finish() # Finish wandb run


def main():
  train(Config(
    learning_starts_at_n_timesteps=10_000,
    log_every_n_steps=1_000,
    save_every_n_steps=50_000,
    wandb_run_name="black-rao-entropy-cpu-20260613-1203",
    resume=True  # Set to True to resume from saved checkpoints
  ))


main()
